<a href="https://colab.research.google.com/github/mkounkel/class_notebooks/blob/main/Lab_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: Large Scale Structure

AST 3414 - Spring 2026

## Introduction

In this lab, you and a partner will analyze a catalog of galaxies. You are tasked with determining how those galaxies are distributed in space (the two-point correlation function) and in brightness (the luminosity function).

### What You'll Learn

1. Compute absolute magnitudes from apparent magnitudes and luminosity distances
2. Construct a galaxy luminosity function and fit a Schechter model to it
3. Interpret the Schechter parameters ($\phi^*$, $M^*$, $\alpha$) physically
4. Measure the angular two-point correlation function $w(\theta)$ of galaxies
5. Explain how galaxy clustering connects to the primordial power spectrum from inflation

---

This is a Pair Programming Lab - Please work together with all members looking at the same copy of the code on one computer. One member will be the Driver who controls the keyboard. The others will be the Navigator(s) who reviews code and think strategically, giving instructions to the Drive.

Every 20-30 minutes an announcement will be made to switch roles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from astropy.coordinates import SkyCoord
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

---

## Part 1: Exploring the SDSS catalog of galaxies

For this lab, you will use a catalog of galaxies that you have previously seen in Galactic Astronomy. The line below will import a catalog of ~290,000 galaxies from the **Sloan Digital Sky Survey (SDSS)**, courtesy of Dr. Kounkel. (Ignore warnings about the nanomaggy unit not being supported by the FITS standard.)

In [ ]:
t=Table.read('https://mkounkel.domains.unf.edu/galactic/sdss_galaxies.fits')

View the table variables, data types, and units.

In [ ]:
t.info

 The catalog conveniently has position in the sky, redshift, apparent magnitudes and luminosity distance.

 This SDSS catalog is able to see galaxies down to a limiting apparent magnitude of about 17.7 in the r-band (wavelengths from 5500 to 7000 Å) and has reliable redshifts out to 0.25. We will therefore apply some qaulity cuts to the galaxy sample to only choose galaxies with known distances, redshifts, and r-band apparent magnitudes in reasonable ranges, which is still most of the sample.

---



In [ ]:
# ── Extract the columns we need ───────────────────────────────────
z_all    = t['redshift']
ra_all   = t['ra']                 # degrees
dec_all  = t['dec']                # degrees
rmag_all = t['rmag']               # apparent r-band magnitude
dL_all   = t['luminosity_distance'] # Mpc

# ── Apply quality cuts ─────────────────────────────────────────────
r_lim = 17.7  # SDSS spectroscopic magnitude limit

good = (
    (z_all > 0.01) &       # remove very nearby galaxies (peculiar velocities dominate)
    (z_all < 0.25) &       # stay within reliable distance range
    (rmag_all > 12) &      # remove saturated sources
    (rmag_all < r_lim) &   # enforce the survey limit
    (dL_all > 0)           # need a valid distance
)

z    = z_all[good]
ra   = ra_all[good]
dec  = dec_all[good]
rmag = rmag_all[good]
dL   = dL_all[good]

print(f'Galaxies after quality cuts: {len(z):,}')

In [ ]:
# ── Visualize the redshift distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(z_all, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Redshift $z$')
axes[0].set_ylabel('Number of galaxies')
axes[0].set_title('Redshift Distribution')

r_lim = 17.7  # SDSS spectroscopic magnitude limit

axes[1].hist(rmag_all, bins=80, color='coral', edgecolor='white', linewidth=0.3)
axes[1].axvline(r_lim, color='k', ls='--', label=f'$r_{{\\rm lim}}$ = {r_lim}')
axes[1].set_xlabel('Apparent $r$ magnitude')
axes[1].set_ylabel('Number of galaxies')
axes[1].set_title('Apparent Magnitude Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

The code below creates a plot in RA, Dec of all galaxies with redshifts between 0.02 and 0.03 in the SDSS survey field.

In [ ]:
toplot = np.where((z>0.02) & (z<0.03))
plt.scatter(ra[toplot], dec[toplot], s=1)
plt.xlabel('RA (deg)')
plt.ylabel('Dec (deg)')
plt.gca().invert_xaxis()
plt.show()

Make your own plots of the distribution of galaxies at different redshift ranges: 0.02-0.03,

In [ ]:
# Your code here

### Discussion Questions

Answer the following with your partner. Write 2-3 sentences for each. When you are done, switch driver and navigator roles for the next part.

**Q1.** Look at the apparent magnitude histogram. Why does the galaxy count drop sharply at $r \approx 17.77$ instead of continuing to rise? What does this tell you about the *true* population of galaxies in the universe?

**Q2.** Describe the distribution of galaxies on the sky. What similarities and differences are there between the nearby universe to the more distant universe?

---

## Part 2: The Galaxy Luminosity Function

The **luminosity function** $\phi(M)$ tells us how many galaxies per unit volume exist at each absolute magnitude. It's one of the most fundamental measurements in extragalactic astronomy.

Recall the **distance modulus** relation:

$$M = m - 5\log_{10}\left(\frac{d_L}{\text{Mpc}}\right) - 25 - K(z)$$

where $K(z)$ is the **K-correction** — an adjustment for the fact that the observed *r*-band filter samples a different rest-frame wavelength at each redshift. For the SDSS *r*-band at low redshift, a good approximation is $K(z) \approx 1.0 \times z$.

*Create a new variable `M_r` that is the calculated r-band absolute magnitude.*

In [ ]:
# Fill in the formula below. You need to use:
#   rmag : apparent r magnitude (array)
#   dL   : luminosity distance in Mpc (array)
#   z    : redshift (array), used for the K-correction K(z) = 1.0 * z

M_r =  # Your code here

print(f'Absolute magnitude range: [{M_r.min():.1f}, {M_r.max():.1f}]')
print(f'Median: {np.median(M_r):.1f}')

For a first look, we'll work with a **single redshift slice** (0.02 < $z$ < 0.06) where we have good sampling across a wide range of absolute magnitudes.

The simplest luminosity function estimator is a **number-count histogram** divided by the survey volume:

$$\phi(M) \approx \frac{N(M)}{V_{\rm survey} \cdot \Delta M}$$

where $V_{\rm survey}$ is the comoving volume of the shell, and $\Delta M$ is the bin width.

To help us calculate comoving distances needed for the comoving volume, we'll use Astropy's cosmology library which takes as in put H$_0$ and $\Omega_m$.


In [ ]:
z_lo, z_hi = 0.02, 0.06
slice = (z >= z_lo) & (z < z_hi)

M_slice = M_r[slice]

print(f'Galaxies in z = [{z_lo}, {z_hi}): {slice.sum():,}')
print(f'M_r range in slice: [{M_slice.min():.1f}, {M_slice.max():.1f}]')

In [ ]:
# The volume of a spherical shell between z_lo and z_hi is:
#   V_shell = (4π/3) * (d_c(z_hi)³ - d_c(z_lo)³)
# But SDSS only covers a fraction f_sky of the sky, so:
#   V_survey = f_sky * V_shell
#
# Use cosmo.comoving_distance(z).value to get d_c in Mpc.

from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

dc_lo = cosmo.comoving_distance(z_lo).value  # comoving distance at z_lo, in Mpc
dc_hi = cosmo.comoving_distance(z_hi).value  # comoving distance at z_hi, in Mpc

f_sky = 8000 / 41253  # SDSS sky fraction

V_shell = (4/3) * np.pi * (dc_hi**3 - dc_lo**3)  # full-sky shell volume
V_survey = V_shell*f_sky  # multiply by f_sky

print(f'Comoving distance at z={z_lo}: {dc_lo:.1f} Mpc')
print(f'Comoving distance at z={z_hi}: {dc_hi:.1f} Mpc')
print(f'Survey volume: {V_survey:.3e} Mpc³')

Now you need to build a histogram of the luminosity function.
1. Create magnitude bins from -25 to -16 with width 0.5 mag.
2. Bin the subset of absolute magnitudes `M_slice` into those bins.
3. Divide by V_survey and the bin width to get φ(M).

Remeber that histograms can be computed using `np.histogram` using something of the form:

```
counts, bin_edges = np.histogram(________, bins=________)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
```


In [ ]:
# Your code here

counts, bin_edges = np.histogram(________, bins=________)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

phi =

Use the code below to plot the luminosity function. We will assume Poisson errors, so that $\sigma_\phi = \frac{\sqrt{N}}{V_{survey} \Delta M}$. You may need to alter the code depending on how you have defined your variables. Add to this plot your Schechter function fit.


In [ ]:
# Poisson error: σ_φ = √N / (V_survey * ΔM)
phi_err = np.sqrt(counts) / (V_survey * bin_width)

# Only keep bins with galaxies
ok = counts > 0

fig, ax = plt.subplots(figsize=(8, 5))

ax.errorbar(bin_centers[ok], np.log10(phi[ok]),
            yerr=0.434 * phi_err[ok] / phi[ok],  # error propagation to log
            fmt='o', capsize=3, markersize=5,
            label='SDSS (0.02 < z < 0.06)')

ax.set_xlabel('$M_r$ (mag)')
ax.set_ylabel(r'$\log_{10}\, \phi$ (Mpc$^{-3}$ mag$^{-1}$)')
ax.set_title('Galaxy Luminosity Function')
ax.invert_xaxis() # bright is to the right (astronomical convention)
ax.legend()
plt.show()

### Questions

Answer the following with your partner. Write 2-3 sentences for each. When you are done, switch driver and navigator roles for the next part.

**Q3.** Describe the shape of the luminosity function in words. Does it look like a Gaussian? A power law? Where does it peak, and what happens at the bright end vs. the faint end?

---

## Part 3: Fit the Schechter function

Schechter proposed a fitting function for the galaxy luminosity function in absolute magnitude:

$$\phi(M) = \frac{2}{5} \ln(10)\; \phi^* \; 10^{\,0.4\,(M^* - M)(\alpha+1)} \; \exp\!\left(-10^{\,0.4\,(M^* - M)}\right)$$

This has **three free parameters**:

| Parameter | Physical meaning |
|-----------|------------------|
| $\phi^*$ | Overall normalization — the number density of galaxies at the "knee" |
| $M^*$ | Characteristic magnitude — the "knee" where the function transitions from power-law to exponential |
| $\alpha$ | Faint-end slope — controls how steeply the number of galaxies rises toward fainter magnitudes |

This is derived from the more familiar version we saw in Ryden in terms of luminosity instead of aboslute magnitudes: $\Phi(L) dL = \Phi^* \big(\frac{L}{L^*}\big)^\alpha \exp \big(- \frac{L}{L^*}\big) \frac{dL}{L^*}$, using the relation $M - M^* = -2.5 \log_{10} (L/L^*)$, where $L^{*}$ is the characteristic luminosity.

Implement the Schechter formula above in terms of the absolute magnitude and the three free parameters listed. It should be a function that returns $\phi(M)$ of the form:

```
def schechter(M, phi_star, M_star, alpha):
    return ________
```

In [ ]:
# Your code here

Use ```scipy.optimize.curve_fit``` to fit your Schechter function to your data and find the best-fit parameters. ```curve_fit``` returns the tuple ```(popt, pcov)``` where `popt` is the best-fit parameters and `pcov` is the covariance matrix. The errors on each parameter are the square root of the diagonal of `pcov`. Look back at past labs or the scipy documentation if you need a refresher.

For an initial guess, try ```p0 = [0.01, -20.5, -1.0]```, where those are the three parameters listed in the table above.

In [ ]:
# Your code here

Now repeat your fit of the Schechter luminosity function for galaxies at higher redshifts between 0.10 < z < 0.15. Be sure to compute the new survey volume which will be different!

Overplot both of your luminosity functions to compare them.

In [ ]:
# Your code here

### Questions

Answer the following with your partner. Write 2-3 sentences for each. When you are done, switch driver and navigator roles for the next part.

**Q4.** Compare the lower- and higher-z luminosity functions. The higher-$z$ bin is missing the entire faint end. Is this because distant galaxies are *actually* more luminous on average, or is something else going on? (Hint: we imposed an apparent magnitude limit of $r < 17.7$ on galaxies at different distances. How does this effect our observations?)